# RL順位と市場人気の「差」の大規模検証

本番の実記録（400レース）で見えた

> **4-6番人気 × 差(人気−RL)が+3以上 → 単勝回収率 165.7%（N=82・13的中・11開催日中7日プラス）**

が本物かどうかを、約2,000レース規模で確かめる。

⚠ **165.7%は3つの異なるモデル（v4/v5/v6）の寄せ集め**で、大半は今は使っていない
モデルが出した順位。現行モデルで測り直さないと採用判断はできない。

## 事前登録した採用基準（結果を見てから変えないこと）

| # | 基準 |
|---|---|
| ① | 単勝回収率 ≥ 105% |
| ② | N ≥ 300 |
| ③ | 期間を前後半に分けても両方100%超 |
| ④ | 開催日ブロックブートストラップの95%CI下限 > 100% |
| ⑤ | 対照群と信頼区間が重ならない |

**1つでも欠けたら「見つからなかった」と結論する。**

## 併せて検証する仮説

本番82頭の内訳では、差の主因別に
スピード能力 302.8%(N=18) / 騎手 186.9%(N=16) / **過去人気推移 10.8%(N=13)**
と分かれた。ただしNが小さく未確定。ここで検証する。

## 実行方法

セル1から順に実行するだけ。セル3の学習に10〜20分かかる。


## セル1: セットアップ

Driveを接続し、GitHubから最新のコードを取得する。

In [ ]:
import os, sys, subprocess, urllib.request, time

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
assert os.path.exists(BASE_DIR), f'BASE_DIRが見つかりません: {BASE_DIR}'
os.chdir(BASE_DIR)
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# 検証用の出力先（本番ファイルと分離）
OUT_DIR = os.path.join(BASE_DIR, 'data', 'backtest')
os.makedirs(OUT_DIR, exist_ok=True)

# ── GitHubから最新コードを取得 ────────────────────────────────
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
FILES = [
    'src/tools/__init__.py', 'src/tools/build_training_data.py', 'src/tools/train_xgb.py',
    'src/features/engine.py', 'src/features/speed_index.py', 'src/features/horse_type.py',
    'src/features/error_tags.py', 'src/features/shap_explain.py',
    'src/utils/config.py', 'src/utils/db.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py', 'src/betting/ev_filter.py',
    'src/betting/app_json.py', 'src/betting/rank_matrix_filter.py',
    'data/course_profiles.json', 'data/course_distance_profiles.json',
    'data/note_schema.json',
]
for rel in FILES:
    dest = os.path.join(BASE_DIR, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    except Exception as e:
        print(f'  ⚠ 取得失敗（既存ファイルを使用）: {rel} — {e}')
for key in [k for k in list(sys.modules) if k.startswith('src')]:
    del sys.modules[key]

# ── データ確認 ───────────────────────────────────────────────
import sqlite3
HIST = os.path.join(BASE_DIR, 'data', 'history.db')
assert os.path.exists(HIST), f'history.dbが見つかりません: {HIST}'
_c = sqlite3.connect(HIST)
_n, _mn, _mx = _c.execute('SELECT COUNT(*), MIN(date), MAX(date) FROM race_history').fetchone()
_h = _c.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
_c.close()
print(f'\n✅ セットアップ完了')
print(f'   history.db : {_n:,}レース / {_h:,}出走')
print(f'   期間       : {_mn} 〜 {_mx}')
print(f'   出力先     : {OUT_DIR}')

# 学習期/検証期の境界（ここを変えれば期間を調整できる）
TRAIN_END  = '2025-12-31'
TEST_START = '2026-01-01'
print(f'\n   学習期: 〜{TRAIN_END} / 検証期: {TEST_START}〜')

## セル2: 学習データ生成

`history.db` から全レース・全馬の特徴量を計算して `horse_features.csv` を作る。

**⚠ ここが一番時間がかかる（20〜50分）。** 途中で切れた場合はこのセルだけ再実行すればよい。

既に最新の `horse_features.csv` がある場合はスキップできる（下の `FORCE_REBUILD` を `False` に）。

In [ ]:
FORCE_REBUILD = True   # 既存CSVを使い回すなら False

import pandas as pd
CSV_PATH = os.path.join(BASE_DIR, 'data', 'horse_features.csv')

_need = FORCE_REBUILD or not os.path.exists(CSV_PATH)
if not _need:
    _tmp = pd.read_csv(CSV_PATH, nrows=1)
    if 'f_popularity' not in _tmp.columns:
        print('⚠ 既存CSVに f_popularity が無い（古い形式）→ 再生成します')
        _need = True

if _need:
    t0 = time.time()
    from src.features.engine import init_engine
    from src.tools.build_training_data import build_training_data
    init_engine(BASE_DIR)
    build_training_data(BASE_DIR)
    print(f'\n⏱ 所要 {(time.time()-t0)/60:.1f} 分')
else:
    print('既存の horse_features.csv を使用します')

df = pd.read_csv(CSV_PATH)
print(f'\n✅ 学習データ: {len(df):,}行 × {len(df.columns)}列')
print(f'   期間: {df["date"].min()} 〜 {df["date"].max()}')
assert 'f_popularity' in df.columns, 'f_popularity が無い。build_training_data の実行を確認'

n_tr = (df['date'] <= TRAIN_END).sum()
n_te = (df['date'] >= TEST_START).sum()
print(f'   学習期: {n_tr:,}行 / 検証期: {n_te:,}行')
assert n_te > 5000, f'検証期のデータが少なすぎます({n_te}行)。TEST_STARTを見直してください'

## セル3: 検証用モデルの学習（2025年末まで）

本番と**同じ設定**（残差学習・同じハイパーパラメータ）で、
**2025年末までのデータだけ**を使って学習する。

本番の `train_xgb()` は本番ファイルを上書きする可能性があるため、
ここでは同等の処理をこのセル内に展開する（本番ファイルには触れない）。

In [ ]:
import numpy as np, xgboost as xgb, json
from sklearn.metrics import roc_auc_score
from src.tools.train_xgb import (_EXCLUDE_COLS, _MARKET_FEAT_COLS,
                                 _popularity_to_base_margin)

t0 = time.time()

# ── 期間分割（検証期の一部を early stopping 用に使う） ───────────
VAL_END = '2026-02-28'          # 学習の early stopping 用
EVAL_START = '2026-03-01'       # ★真のout-of-sample（フィルタ検証はここだけ使う）

train_df = df[df['date'] <= TRAIN_END].copy()
val_df   = df[(df['date'] >= TEST_START) & (df['date'] <= VAL_END)].copy()
eval_df  = df[df['date'] >= EVAL_START].copy()
print(f'学習 {len(train_df):,}行 / 早期停止用 {len(val_df):,}行 / 検証 {len(eval_df):,}行')

# ── 特徴量列（本番と同じ規則: 除外列 + f_popularity を除く数値列）──
feat_cols = [c for c in df.columns
             if c not in _EXCLUDE_COLS and c not in _MARKET_FEAT_COLS
             and df[c].dtype in ('float64', 'int64', 'float32', 'int32')]
print(f'特徴量数: {len(feat_cols)}')

def _prep(d):
    d = d.copy()
    d['_n_horses'] = d.groupby('race_id')['horse_num'].transform('count')
    pop = d['f_popularity'].fillna(d['_n_horses'] / 2)
    bm = _popularity_to_base_margin(pop, d['_n_horses'])
    X = d[feat_cols].fillna(5.0)
    return d, X, bm

train_df, X_tr, bm_tr = _prep(train_df)
val_df,   X_va, bm_va = _prep(val_df)
eval_df,  X_ev, bm_ev = _prep(eval_df)
y_tr, y_va = train_df['is_fukusho'], val_df['is_fukusho']

pos_rate = y_tr.mean()
spw = round((1 - pos_rate) / max(pos_rate, 0.01), 2)

dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=feat_cols); dtr.set_base_margin(bm_tr)
dva = xgb.DMatrix(X_va, label=y_va, feature_names=feat_cols); dva.set_base_margin(bm_va)

params = {'objective': 'binary:logistic', 'max_depth': 6, 'learning_rate': 0.05,
          'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 10,
          'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': spw,
          'eval_metric': 'logloss', 'seed': 42, 'nthread': -1}
booster = xgb.train(params, dtr, num_boost_round=500,
                    evals=[(dva, 'val')], early_stopping_rounds=50, verbose_eval=100)

# ⚠ output_margin=True 必須（無いと2重sigmoidになる。2026-07-27に本番で発覚したバグ）
va_margin = booster.predict(dva, output_margin=True)
print(f'\n✅ 学習完了  Val AUC: {roc_auc_score(y_va, 1/(1+np.exp(-va_margin))):.4f}')
print(f'⏱ 所要 {(time.time()-t0)/60:.1f} 分')

booster.save_model(os.path.join(OUT_DIR, 'backtest_model.pkl'))
with open(os.path.join(OUT_DIR, 'backtest_feature_cols.json'), 'w') as f:
    json.dump({'feature_cols': feat_cols, 'residual': True, 'train_end': TRAIN_END}, f)
print(f'   検証用モデルを保存: {OUT_DIR}/backtest_model.pkl（本番ファイルは無変更）')

## セル4: 検証期の推論 → rl_rank 算出

検証期（2026-03以降）の全馬を推論し、レース内でのAI評価順位（rl_rank）を出す。

さらに `history.db` から **確定人気** と **単勝配当** を結合する
（`win_odds` は0%充足のため使えないが、単勝ROIの計算には
「勝ったか」＋「勝った時の配当」があれば足りる）。

In [ ]:
t0 = time.time()

dev = xgb.DMatrix(X_ev, feature_names=feat_cols); dev.set_base_margin(bm_ev)
eval_df['raw_margin'] = booster.predict(dev, output_margin=True)

# レース内でmargin降順 → rl_rank（1位が最高評価）
eval_df['rl_rank'] = eval_df.groupby('race_id')['raw_margin'] \
                            .rank(ascending=False, method='first').astype(int)

# ── history.db から確定人気・着順・単勝配当を結合 ──────────────
conn = sqlite3.connect(HIST)
hh = pd.read_sql_query(
    'SELECT race_id, horse_num, popularity, place, tansho_payout FROM horse_history', conn)
conn.close()
hh['horse_num'] = pd.to_numeric(hh['horse_num'], errors='coerce')
eval_df['horse_num'] = pd.to_numeric(eval_df['horse_num'], errors='coerce')

m = eval_df.merge(hh, on=['race_id', 'horse_num'], how='left', suffixes=('', '_h'))
print(f'結合前 {len(eval_df):,}行 → 結合後 {len(m):,}行')

# 人気が有効な馬のみ対象（99や欠損は除外）
m = m[(m['popularity'].notna()) & (m['popularity'] >= 1) & (m['popularity'] < 99)].copy()
m['popularity'] = m['popularity'].astype(int)
m['won'] = (m['place'] == 1)
m['payout'] = m['tansho_payout'].fillna(0)
# 勝ったのに配当が取れていない行は集計から外す（回収率を過小評価しないため）
bad = m['won'] & (m['payout'] <= 0)
print(f'⚠ 勝ったが配当欠損のため除外: {bad.sum()}頭')
m = m[~bad].copy()

print(f'\n✅ 検証対象: {len(m):,}頭 / {m["race_id"].nunique():,}レース')
print(f'   期間: {m["date"].min()} 〜 {m["date"].max()}')
print(f'   全体の単勝回収率（全馬に賭けた場合）: {m["payout"].sum()/(len(m)*100)*100:.1f}%')
print(f'⏱ 所要 {(time.time()-t0):.0f} 秒')

## セル5: 差(gap)の算出と SHAP 分解

In [ ]:
# == 差(gap)の算出 + SHAP分解 =============================================
# rl_rank はレース内の raw_margin 降順。
# ⚠ 本番の rl_rank は total（相対ブレンド・ペースボーナス・エラータグ補正込み）
#    の順位だが、相対ブレンドはレース内の一次変換なので順位を変えない。
#    順位を動かすのはペースボーナスとエラータグ補正だけで、影響は小さい。
#    ここでは raw_margin 順位で近似する（この近似の限界は最後に報告する）。
import numpy as np, pandas as pd
from src.features.shap_explain import FEATURE_CATEGORY_MAP

t0 = time.time()
m['gap'] = m['popularity'] - m['rl_rank']          # +ならAIが市場より強気

# ── 差をカテゴリに分解 ────────────────────────────────────────────
contribs = booster.predict(dev, pred_contribs=True)      # (n, 特徴量数+1)
cat_of = {c: FEATURE_CATEGORY_MAP.get(c, 'その他') for c in feat_cols}
cats = sorted(set(cat_of.values()))
tmp = pd.DataFrame(0.0, index=eval_df.index, columns=cats)
for i, c in enumerate(feat_cols):
    tmp[cat_of[c]] += contribs[:, i]
tmp['race_id'] = eval_df['race_id'].values
tmp['horse_num'] = pd.to_numeric(eval_df['horse_num'], errors='coerce').values
m = m.merge(tmp, on=['race_id', 'horse_num'], how='left')

m['主因'] = m[cats].idxmax(axis=1)
print(f'差の分解 完了  {len(m):,}頭 / {m.race_id.nunique():,}レース  '
      f'({time.time()-t0:.0f}秒)')
print(f'\n差の分布: 中央値 {m.gap.median():.0f} / '
      f'+3以上 {(m.gap>=3).sum():,}頭 / 4-6番人気かつ+3以上 '
      f'{((m.popularity.between(4,6))&(m.gap>=3)).sum():,}頭')

# ── ★ 先に「基準②(N>=300)に届く見込みがあるか」を確認する ──────────
# 本番の実記録では 400レースで82頭（1レースあたり0.21頭）出ていた。
# ここでの出現率がそれより大幅に低い場合、レース数を増やしてもNは足りない。
_n_tgt = int(((m.popularity.between(4, 6)) & (m.gap >= 3)).sum())
_rate = _n_tgt / max(m.race_id.nunique(), 1)
print(f'\n1レースあたりの該当頭数: {_rate:.3f}頭  （本番実記録では 0.21頭）')
if _n_tgt < 300:
    print(f'⚠ 該当が {_n_tgt}頭しかなく、事前登録した基準②(N>=300)には届かない。')
    print('  → この時点で「採用しない」はほぼ確定。ただし基準を後から緩めないこと。')
    print('  　 セル6以降は「向きだけでも見ておく」ための参考として実行する。')


## セル6: ★ 本命の検証（事前登録した基準で判定）

In [ ]:
# == ★ 本命の検証: 4-6番人気 × 差+3以上 ==================================
import numpy as np
rng = np.random.default_rng(0)
m['day'] = m['race_id'].str[:8]      # ⚠ date列は 'YYYY-MM-DD' なので race_id から取る

def blockboot(s, n=4000):
    """開催日ブロックブートストラップ。同日レースは条件が相関するため必須。

    開催日が3日以下だとブロックが足りず区間推定が成立しない
    （2026-07-27に59点を独立サンプル扱いして過信した失敗の再発防止）。
    """
    days = s.day.unique()
    if len(s) == 0 or len(days) < 4:
        return float('nan'), float('nan')
    by = {k: v.payout.values for k, v in s.groupby('day')}
    out = [np.concatenate([by[k] for k in rng.choice(days, len(days), replace=True)]).mean()
           for _ in range(n)]
    return np.percentile(out, [2.5, 97.5])


def _gt(a, b):
    """NaN（区間推定できず）は「基準を満たした」と誤判定しないようFalseにする。"""
    return bool(a == a and b == b and a > b)

tgt = m[(m.popularity.between(4, 6)) & (m.gap >= 3)]
ctl = m[(m.popularity.between(4, 6)) & (m.gap < 3)]

print('=' * 64)
print(f'対象: 4-6番人気 × 差+3以上')
print(f'  N={len(tgt):,}  勝率{100*tgt.won.mean():.1f}%  回収率{tgt.payout.mean():.1f}%  '
      f'開催日{tgt.day.nunique()}日  的中{int(tgt.won.sum())}本')
print(f'対照群: 4-6番人気 × 差+2以下')
print(f'  N={len(ctl):,}  勝率{100*ctl.won.mean():.1f}%  回収率{ctl.payout.mean():.1f}%')
print('=' * 64)

res = {}
res['①回収率>=105%'] = (tgt.payout.mean() >= 105, f'{tgt.payout.mean():.1f}%')
res['②N>=300'] = (len(tgt) >= 300, f'{len(tgt)}')

days = sorted(tgt.day.unique()); mid = days[len(days)//2]
a, b = tgt[tgt.day <= mid], tgt[tgt.day > mid]
res['③前後半とも100%超'] = (a.payout.mean() > 100 and b.payout.mean() > 100,
                       f'前半{a.payout.mean():.1f}% / 後半{b.payout.mean():.1f}%')

lo, hi = blockboot(tgt)
res['④CI下限>100%'] = (_gt(lo, 100), f'95%CI [{lo:.1f}, {hi:.1f}]')
clo, chi = blockboot(ctl)
res['⑤対照群とCI非重複'] = (_gt(lo, chi), f'対照群CI [{clo:.1f}, {chi:.1f}]')

print('\n【事前登録した採用基準の判定】')
for k, (ok, detail) in res.items():
    print(f'  {"✅" if ok else "❌"} {k:20s} {detail}')
verdict = all(v[0] for v in res.values())
print('\n' + ('★ 全基準クリア → 採用を検討してよい' if verdict
              else '→ 基準未達。採用しない（「見つからなかった」が結論）'))

print('\n【開催日ごとの成績】（1〜2日に偏っていないか）')
g = tgt.groupby('day').agg(N=('payout','size'), 的中=('won','sum'), 回収率=('payout','mean'))
print(g.round(1).to_string())
print(f'  プラスだった日: {(g.回収率>100).sum()}/{len(g)}日')

w = tgt[tgt.won].sort_values('payout', ascending=False)
if len(w) >= 3:
    print(f'\n【高配当依存の確認】最高配当を除くと '
          f'{tgt[~tgt.index.isin(w.index[:1])].payout.mean():.1f}% / '
          f'上位3本を除くと {tgt[~tgt.index.isin(w.index[:3])].payout.mean():.1f}%')


## セル7: 差の「中身」で絞り込めるか

In [ ]:
# == 差の「中身」で絞り込めるか ===========================================
# 本番82頭では スピード能力302.8% / 過去人気推移10.8% と分かれた（N小・未確定）。
print('【差の主因ごとの成績】4-6番人気 × 差+3以上 の中で\n')
t = tgt.groupby('主因').agg(N=('payout','size'), 的中=('won','sum'),
                          勝率=('won','mean'), 回収率=('payout','mean'))
t['勝率'] = (t['勝率']*100).round(1); t['回収率'] = t['回収率'].round(1)
t['的中'] = t['的中'].astype(int)
_shown = t[t.N >= 30].sort_values('回収率', ascending=False)
print(_shown.to_string() if len(_shown)
      else f'  主因別に30頭以上あるグループなし（対象が{len(tgt)}頭しかない）')

print('\n\n【的中馬 vs 不的中馬 のカテゴリ寄与】')
if tgt.won.sum() >= 5:
    cmp = pd.DataFrame({'的中馬': tgt[tgt.won][cats].mean(),
                        '不的中馬': tgt[~tgt.won][cats].mean()})
    cmp['差'] = cmp['的中馬'] - cmp['不的中馬']
    print(cmp.sort_values('差', ascending=False).round(4).to_string())
else:
    print(f'  的中が{int(tgt.won.sum())}本しかなく比較できない')

print('\n\n【仮説の検証】主因で絞ると基準を満たすか')
_cands = t[t.N >= 100].index
for name in _cands:
    s = tgt[tgt.主因 == name]
    lo2, hi2 = blockboot(s)
    ok = (s.payout.mean() >= 105 and len(s) >= 300 and _gt(lo2, 100))
    print(f'  {"✅" if ok else "❌"} 主因={name:10s} N={len(s):4d} '
          f'回収率{s.payout.mean():6.1f}%  95%CI [{lo2:6.1f}, {hi2:6.1f}]')
if not len(_cands):
    print('  100頭以上ある主因なし（絞り込みの検証はできない）')
print('\n⚠ 主因で切ると必ずNが減る。②N>=300 を満たせるかが分かれ目。')


## セル8: 頑健性チェックと限界

In [ ]:
# == 頑健性: 差の閾値・人気帯を変えても成り立つか =========================
print('【差の閾値を変える】4-6番人気\n')
for th in (1, 2, 3, 4, 5):
    s = m[(m.popularity.between(4, 6)) & (m.gap >= th)]
    if len(s) < 30: continue
    lo3, hi3 = blockboot(s)
    print(f'  差>=+{th}: N={len(s):5,d} 勝率{100*s.won.mean():5.1f}% '
          f'回収率{s.payout.mean():6.1f}%  CI [{lo3:6.1f}, {hi3:6.1f}]')

print('\n【人気帯を変える】差+3以上\n')
for lo_p, hi_p, lab in [(1,3,'1-3番人気'), (4,6,'4-6番人気'),
                        (7,9,'7-9番人気'), (10,18,'10番人気以上')]:
    s = m[(m.popularity.between(lo_p, hi_p)) & (m.gap >= 3)]
    if len(s) < 30: continue
    lo4, hi4 = blockboot(s)
    print(f'  {lab:12s}: N={len(s):5,d} 勝率{100*s.won.mean():5.1f}% '
          f'回収率{s.payout.mean():6.1f}%  CI [{lo4:6.1f}, {hi4:6.1f}]')

print('\n\n【この検証の限界】')
print('  ・人気は history.db の**確定人気**。本番は朝の人気を使うため条件が違う')
print('    （実測で平均1.5位ずれ、1番人気が入れ替わるレースが34.9%ある）')
print('  ・rl_rank は raw_margin 順位で近似（ペースボーナス・エラータグ補正を除く）')
print('  ・検証期のモデルは2025年末までで学習したもの。本番モデルとは別物')
print('  → 「効果の向き」は信頼できるが、「回収率の絶対値」はそのまま本番に移らない')
